# 01 · Quickstart

환경이 제대로 구축됐는지 확인하고, `src` 모듈 사용법을 익히는 노트북.

커널이 **Python (vsc_workspace)** 인지 우측 상단에서 확인하세요.

In [ ]:
# 프로젝트 루트를 import 경로에 추가 (노트북이 notebooks/ 안에 있으므로)
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 모듈을 고치면 커널 재시작 없이 반영
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt

from src import config, data_loader as dl, indicators as ind, plotting

font = plotting.setup()
print(f"pandas {pd.__version__} · 한글 폰트: {font}")

## 1. 지수 가져오기 — KOSPI / NASDAQ

In [ ]:
kospi = dl.get_price("KOSPI", "2020-01-01")
nasdaq = dl.get_price("NASDAQ", "2020-01-01")

print("KOSPI ", kospi.shape, kospi.index.min().date(), "~", kospi.index.max().date())
print("NASDAQ", nasdaq.shape)
kospi.tail()

## 2. 성과 요약

`ind.summary()` 로 수익률·변동성·샤프·MDD를 한 번에.

In [ ]:
pd.DataFrame({
    "KOSPI": ind.summary(kospi["Close"]),
    "NASDAQ": ind.summary(nasdaq["Close"]),
})

## 3. 여러 종목 종가 한 번에

시장이 다르면 휴장일이 달라 NaN이 생긴다 → 비교할 땐 `.dropna()`.

In [ ]:
closes = dl.get_closes(["KOSPI", "NASDAQ", "005930", "AAPL"], "2023-01-01").dropna()

# 시작점을 100으로 맞춰 상대 성과 비교
normalized = closes / closes.iloc[0] * 100

ax = normalized.plot(title="상대 성과 비교 (시작 = 100)")
ax.set_ylabel("지수화 가격")
ax.set_xlabel("")
plt.show()

## 4. 기술적 지표

In [ ]:
samsung = dl.get_price("005930", "2024-01-01")
enriched = ind.add_all(samsung)
enriched[["Close", "sma20", "sma60", "rsi14", "macd", "drawdown"]].tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

enriched[["Close", "sma20", "sma60"]].plot(ax=ax1, title="삼성전자 종가와 이동평균")
ax1.set_xlabel("")

enriched["rsi14"].plot(ax=ax2, color="#7c3aed")
ax2.axhline(70, ls="--", lw=0.8, color="gray")
ax2.axhline(30, ls="--", lw=0.8, color="gray")
ax2.set_ylabel("RSI(14)")
ax2.set_xlabel("")

plt.tight_layout()
plt.show()

## 5. 종목 검색

In [ ]:
dl.find_symbol("삼성").head(10)

---
캐시를 비우려면 `dl.clear_cache()`.